This document outlines the process of converting the raw text from each pdf in the "briefs" and "opinions" folders into 1) dictionaries that map the docket number of the document to its raw text, 2) csv files containing the dictionary data, and 3) a pkl file containing that data.

In [3]:
import pdfplumber
import pickle
import pandas as pd
import numpy as np
import os
import re
import csv

In [8]:
def docket_to_csv(input_pathname, output_filename, pickle_filename):
    '''
    Extracts the text from pdfs in a folder (indicated by input_pathname), 
    converts the text to a csv file and returns a dictionary mapping 
    the docket number for the document to its text content
    Inputs:
        input_pathname: a string indicating the location of the pdf to be converted
        output_filename: a string indicating the name of the output csv
        pickle_filename: a string indicating the name of the pickle file the dictionary will be saved as 
    '''
    docket_dict = {}
    
    # loop through each filename in the input folder
    for filename in os.listdir(input_pathname):
        if filename.endswith('.pdf'): # if it is a pdf, use pdfplumber to extract the text
            pdf_path = os.path.join(input_pathname, filename)
            with pdfplumber.open(pdf_path) as pdf:
                text = ''
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:  # avoid None values
                        text += page_text + '\n'
        
                docket_pattern = r'(\d\d\-\d+)'  # extract docket number as identifier
                match = re.search(docket_pattern, filename)
                if match:
                    docket_num = match.group()
                    entry = {'filename':filename, 'text':text}
                    if docket_num in docket_dict:
                        docket_dict[docket_num].append(entry)
                    else:
                        docket_dict[docket_num] = [entry]
                else:
                    print(f"Skipping file {filename} due to no docket number match.")

    print(f"Extracted text from {len(docket_dict.keys())} docket numbers.")
    
    # save dictionary as pkl file to avoid having to run this function again
    with open(pickle_filename, "wb") as f: 
            pickle.dump(docket_dict, f)

    # export extracted data to csv to be used in preprocessing 
    with open(output_filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Docket Number", "Filename", "Extracted Text"])  # Add header
        for docket_num, entries in docket_dict.items():
            for entry in entries:
                writer.writerow([docket_num, entry["filename"], entry["text"]])

First, I will convert the amicus briefs to raw text (the output can be found in raw_text.csv)

In [9]:
# don't run again!!!
ab_path = '/Users/haileyhansen/MACSS/30100/macs-30100-hailhan/P3_individual_project/briefs'
docket_to_csv(ab_path, 'raw_text_filename.csv', 'ab_dict.pkl')

Extracted text from 98 docket numbers.


Second, I will convert the opinions to raw text (the output can be found in raw_op_text.csv)

In [4]:
# don't run again!!
op_path = '/Users/haileyhansen/MACSS/30100/macs-30100-hailhan/P3_catholic_kmeans/opinions'
docket_to_csv(op_path, 'raw_op_text.csv', 'op_dict.pikl')

Skipping file 84_1970_usr_5000.pdf due to no docket number match.
Skipping file 21a24_pc_0006.pdf due to no docket number match.
Skipping file 20a34_pc_0011.pdf due to no docket number match.
Skipping file 18a774_oro_0005.pdf due to no docket number match.
Skipping file 534_1970_usr_5000.pdf due to no docket number match.
Skipping file 21a85_oro_0016.pdf due to no docket number match.
Extracted text from 173 docket numbers.


Notice that a few files above were skipped: this is likely because many of the pdfs were scans of original documents, which makes them difficult for pdfplumber to interpret. This is not an issue in the grand scheme of the project: I have more than enough data for my task, and I only ended up using the dockets that had both a brief and an opinion match anyway.